# List-aware · broader individual-model and feature-pool screen

**Stage A before the list-aware ensemble factory.** The new `curated` suite carries forward promising features from the 1,050-home individual and composite screens without claiming those results generalize. Compare **a curated three-family feature-recipe suite (or the original 44-model catalog)** (the entire catalog) in a factorized research screen. The goal is a **diverse candidate pool**, *not* choosing an ensemble or freezing any feature group yet.

The experiment distinguishes:

- Model families / objectives: CatBoost MAE, Huber, RMSE; tuned XGBoost and LightGBM, direct or sale/list ratio targets.
- Text: no text, original TF-IDF, selected binary, training-fitted full binary/count/TF-IDF.
- Comparable sales: original, age-first, same-property-type quota, both.
- Spatial additions: splines alone, geography alone, both, and prior AVM designs.

**Screen:** Mar–May 2025, 10,000-row/350-home/300-tree pilot defaults, monthly expanding origins.
**Confirmation:** Jun–Jul 2025, full legal history/complete months/original tree counts, ONLY the top capped shortlist chosen from the screen.

**Important:** historical months have been studied previously; these are exploratory development results. Do **not** compare pilot MdAPE directly with the earlier **3.8996%** full-strength single-fit fixed-origin Aug–Oct reference—different months, fits and budgets.

The notebook delegates fitting and caching to `feature/listaware_pool_screen.py` and **does not** change the production models or off-market factory.


## 1 · Find the checkout and load the experiment catalog

Works from `experiments/` or the repository root. You may set `DATA_DASH_ROOT` to the checkout. Keep your listings Parquet/CSV source unchanged between runs.


In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

start = Path(os.environ.get("DATA_DASH_ROOT") or Path.cwd()).expanduser().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "feature" / "listaware_pool_screen.py").is_file()
             and (p / "data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the repository or set DATA_DASH_ROOT.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from feature.listaware_pool_screen import (
    CATALOG, GROUPS, SUITES, SCREEN_MONTHS, CONFIRM_MONTHS,
    SEED_REFERENCE, catalog_frame, choose_confirmation, run,
)
print("Repository:", ROOT)
print("Candidates in full catalog:", len(CATALOG))
display(pd.DataFrame([
    {"suite": suite, "members": len(names)} for suite, names in SUITES.items()
]))


## 2 · Configure the screen and later confirmation

Start with `SUITE="core"` for a quick pipeline check, then use `"curated"` for the pooled-feature comparison. The previous `"all"` catalog remains available and unchanged. Use different output directories across suites. Full-budget selection then uses a 14-member shortlist with predeclared family/feature anchors.

`RUN_SCREEN` and `RUN_CONFIRM` default to **False**, so opening or executing this notebook does not unexpectedly train dozens of models. Rerun with identical settings to resume cached candidate/month predictions.


In [ ]:
SUITE = "curated"    # curated | core | text | comparables | spatial | objectives | avm | all
THREADS = 4
SEED = 42
BOOTSTRAP_REPEATS = 300

SCREEN_TRAIN_CAP = 10_000
SCREEN_VALIDATION_CAP = 350
SCREEN_TREE_CAP = 300

TOP_K = 14  # 4 historical controls + selected Cat/XGB/LGB feature anchors + best screen performers
RUN_TESTS = False
RUN_SCREEN = False
RUN_CONFIRM = False

SCREEN_DIR = ROOT / "outputs" / f"listaware_pool_screen_{SUITE}"
CONFIRM_DIR = ROOT / "outputs" / f"listaware_pool_confirmation_{SUITE}"

print("Screen:", SCREEN_DIR)
print("Confirmation:", CONFIRM_DIR)
print("Pilot folds:", SCREEN_MONTHS)
print("Later full-strength folds:", CONFIRM_MONTHS)
print("Candidates:", len(SUITES[SUITE]))


## 3 · Candidate catalog

Every screen compares the same held-out homes and includes `cat_huber_baseline` as a matched reference. All candidates use the existing list-aware data-hiding rules; the original 50/35/15 blend is **not** refitted or reweighted in this individual-model screen.

There is deliberately **no** feature bagging, row bootstrapping or Optuna here: first identify which feature/model configurations merit becoming ingredients of those next-stage algorithms.


In [ ]:
candidate_catalog = catalog_frame(SUITES[SUITE])
display(candidate_catalog)
display(candidate_catalog.groupby(["family", "target"]).size().rename("candidate_count"))


## 4 · Optional fast safety tests

The tests use synthetic listings and stub model fits to check candidate registration, exact month/home matching, no answer leakage, strict cache manifests, resume, and deterministic shortlist selection. They do not run a full housing model.


In [ ]:
if RUN_TESTS:
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pytest",
        "tests/test_listaware_pool_screen.py",
        "tests/test_listaware_shotgun.py",
        "-q",
    ], check=True, cwd=ROOT)
else:
    print("Tests skipped; set RUN_TESTS=True to validate the setup.")


## 5 · Run or resume the bounded chronological screen (March–May)

`RUN_SCREEN=True` fits each candidate on sale histories that end **before that month's first day**, then validates on that month's sales with sold price and post-sale fields hidden.

Outputs and each completed model-month forecast are written incrementally; rerun after interruption with **unchanged settings**, not a new candidate set in the same output directory.


In [ ]:
screen_pooled = screen_monthly = screen_types = screen_predictions = None

if RUN_SCREEN:
    screen_pooled, screen_monthly, screen_types, screen_predictions = run(
        stage="screen", suite=SUITE, output_dir=SCREEN_DIR,
        train_cap=SCREEN_TRAIN_CAP, validation_cap=SCREEN_VALIDATION_CAP,
        tree_cap=SCREEN_TREE_CAP, threads=THREADS, seed=SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
elif (SCREEN_DIR / "pooled_scores.csv").is_file():
    screen_record = json.loads((SCREEN_DIR / "manifest.json").read_text())
    if (screen_record.get("stage") != "screen"
        or set(screen_record.get("candidates", ())) != set(SUITES[SUITE])
        or tuple(screen_record.get("months", ())) != SCREEN_MONTHS):
        raise ValueError("This screen directory belongs to a different protocol.")
    screen_pooled = pd.read_csv(SCREEN_DIR / "pooled_scores.csv")
    screen_monthly = pd.read_csv(SCREEN_DIR / "monthly_scores.csv")
    screen_types = pd.read_csv(SCREEN_DIR / "house_type_scores.csv")
    print("Loaded saved screen reports; no training.")
else:
    print("No finished screen; set RUN_SCREEN=True in section 2.")


## 6 · Inspect individual model strength and error stability

`delta_mdape_pp` uses a matched CatBoost Huber reference **on the same screen folds**, not the earlier 3.8996% final-stage blend. Negative deltas favour the challenger.

Compare MdAPE, MAPE, P90 APE and all monthly folds. A model can be retained later for diversity even if it is not the best standalone performer.


In [ ]:
if screen_pooled is not None:
    screen_pooled = screen_pooled.copy()
    screen_pooled["screen_objective"] = .65*screen_pooled.mdape + .35*screen_pooled.mape
    view = screen_pooled.sort_values(["screen_objective", "mdape"])
    display(view[[
        "variant", "homes", "mdape", "mape", "p90_ape",
        "delta_mdape_pp", "delta_mape_pp", "screen_objective",
    ]].round(4).reset_index(drop=True))

    display(Markdown("**Monthly MdAPE (lower is better)**"))
    display(screen_monthly.pivot_table(
        index="variant", columns="month", values="mdape"
    ).reindex(view.variant).round(4))
else:
    print("Run or load screen results first.")


## 7 · Freeze a capped individual-model shortlist using ONLY the screen

The selection rule combines 65% MdAPE and 35% MAPE, ensures CatBoost/XGBoost/LightGBM representation, and includes the old list-aware controls. For the `curated` suite, additional feature anchors (selected-binary spatial, full-binary spatial, XGB splines, LGB splines and LGB advanced) are also retained. It reads **no** June–July labels.

You can inspect the proposed shortlist here. The confirmation runner reconstructs it from the saved screen results and stores their SHA-256 fingerprints so the selection cannot change silently after confirmation.


In [ ]:
shortlist = None
if screen_pooled is not None:
    from models.listaware_candidate_pool import CONFIRMATION_ANCHORS
    shortlist = choose_confirmation(
        screen_pooled, top_k=TOP_K,
        anchors=CONFIRMATION_ANCHORS if SUITE == "curated" else (),
    )
    display(catalog_frame(shortlist))
    print("Confirming", len(shortlist), "candidates on FULL history.")
else:
    print("A completed pilot screen is required before choosing confirmation candidates.")


## 8 · Full-strength chronological confirmation (June–July)

**Only enable `RUN_CONFIRM` after you are satisfied with the stage-A shortlist.** Confirmation refits selected candidates monthly, on every eligible completed sale before that month, with the original complete iteration counts and all held-out homes.

Its validation months are chronologically later than the screening months. They are *not* a never-inspected holdout for the project, so report uncertainty accordingly. Candidate selection and hyperparameters are not adjusted based on these confirmation results.


In [ ]:
confirm_pooled = confirm_monthly = confirm_types = confirm_predictions = None

if RUN_CONFIRM:
    if shortlist is None:
        raise RuntimeError("Finish and load stage-A screen scores first.")
    confirm_pooled, confirm_monthly, confirm_types, confirm_predictions = run(
        stage="confirm", suite=SUITE,
        screen_dir=SCREEN_DIR, output_dir=CONFIRM_DIR, top_k=TOP_K,
        threads=THREADS, seed=SEED, bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
elif (CONFIRM_DIR / "pooled_scores.csv").is_file():
    confirm_record = json.loads((CONFIRM_DIR / "manifest.json").read_text())
    if confirm_record.get("stage") != "confirm" or tuple(
        confirm_record.get("months", ())
    ) != CONFIRM_MONTHS:
        raise ValueError("Confirmation output does not match expected months/stage.")
    confirm_pooled = pd.read_csv(CONFIRM_DIR / "pooled_scores.csv")
    confirm_monthly = pd.read_csv(CONFIRM_DIR / "monthly_scores.csv")
    print("Loaded full-strength confirmation results; no training.")
else:
    print("No completed confirmation; set RUN_CONFIRM=True when ready.")


## 9 · Compare pilot and full-strength rankings (not raw accuracy)

Pilot and confirmation differ in **months, training-row counts, evaluation-row counts, and boosting rounds**, so comparing their absolute MdAPE as an apparent gain is misleading. Instead inspect whether the candidates' **relative ranking and within-fold month stability** persist under the stronger protocol.

Beware pilot-to-full reversal: we have already observed that in this project.


In [ ]:
if screen_pooled is not None and confirm_pooled is not None:
    screen_view = screen_pooled[["variant", "mdape", "mape"]].rename(
        columns={"mdape": "screen_mdape", "mape": "screen_mape"}
    )
    full_view = confirm_pooled[["variant", "mdape", "mape", "p90_ape"]].rename(
        columns={"mdape": "confirm_mdape", "mape": "confirm_mape"}
    )
    compared = full_view.merge(screen_view, on="variant", how="left",
                               validate="one_to_one")
    compared["screen_rank"] = compared.screen_mdape.rank(method="min")
    compared["confirm_rank"] = compared.confirm_mdape.rank(method="min")
    display(compared.sort_values("confirm_rank").round(4))
else:
    print("Both screen and confirmation scoreboards are needed.")


## 10 · What follows Stage A

**Do not freeze list-aware features after this screen.** Use the confirmed strong configurations, along with a few error-diverse alternatives, as candidates for:

1. **List-aware feature bagging:** selected vs additional BoW terms, weighted/random feature subsets, and multiple seeds.
2. **Row bootstrap/subbagging:** fit comparable-sale context and text vocabulary on legally eligible **full** historical training folds *before* sampling estimator rows.
3. **Model-specific Optuna:** tune CatBoost, XGBoost, and LightGBM on earlier chronological folds.
4. **Separate list-aware ensemble factory:** choose membership and weights based on marginal gain on separate selection folds and a frozen later-period audit.

The off-market factory remains independent, and the existing 50/35/15 list-aware blend remains a useful benchmark. No submission or saved production model is changed by this notebook.
